# Federated Orthogonal Training (FOT)
### ResNet18 on CIFAR-100 + α = 0.1

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""

Settings:
    - NUM_CLIENTS = 10
    - Dirichlet non-IID with ALPHA = 0.1
    - EPOCHS (global rounds) = 200
    - LOCAL_EPOCHS = 5

Logs per round:
    - acc_train (global on union of all client train data)
    - acc_test
    - round_time_sec
    - mean_FgT_loss (window-based forgetting)
"""

import math
import random
import time
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models



# ============================= Parameters ============================== #
ALPHA = 0.1                 # Dirichlet non-IID concentration
PROGRAM_NAME = "FOT - CIFAR-100 α=0.1"
DATASET_NAME = "CIFAR100"
DATA_ROOT = "data"


EPOCHS = 200                # Global communication rounds
LOCAL_EPOCHS = 5            # Local SGD epochs per client per round
NUM_CLIENTS = 10
FRAC = 1.0                  # Fraction of clients per round


# --------------------------- Optimization ----------------------------- #

BATCH_SIZE = 256
LR_BASE = 0.001
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
LABEL_SMOOTH = 0.0          # 0.0 → standard CE
NUM_WORKERS = 4
SEED = 1234

# Turn off cosine. Only use constant LR 
cosine_schedule = False
if cosine_schedule:
    SCHEDULER = "cosine"               # Making it the same shape across runs
else:
    print(f"Using constant LR = {LR_BASE} (cosine_schedule={cosine_schedule})")


# ---------------------------- Orthogonal ------------------------------ #

ORTH = True                 # True → FOT-style projection, False → plain FedAvg
TOP_R = 8                   # Max basis rank per layer
WARMUP_ROUNDS = 10          # No projection during warmup; only build basis
REBUILD_EVERY = 0           # For logging only (kept to match your CSV schema)

# ---------------------------- Forgetting ------------------------------ #

NUM_FGT_WINDOWS = 5         # Number of windows over test set for forgetting


# To print in color -------test/train of the client side
def prRed(skk): print("\033[91m {}\033[00m" .format(skk)) 
def prGreen(skk): print("\033[92m {}\033[00m" .format(skk))

    
# ====================== Confirm =======================================
cut_at = "Original"
prRed(f"\tData:\t\t{DATASET_NAME}\n\tCut:\t\t{cut_at}\n\tGlobal Rounds:\t{EPOCHS}\n\tClients:\t{NUM_CLIENTS}\n\tLocal Epochs:\t{LOCAL_EPOCHS}\n\tα:\t\t{ALPHA}\n\tWarm-up:\t{WARMUP_ROUNDS}\n\tRebuild:\tN/A")


# ============================================================================
#                         Colored terminal output
# ============================================================================
# To print in color -------test/train of the client side

# After printing color it has to reset back to default text color
#RESET = '\033[0m'
BRIGHT_YELLOW = '\033[93m'
RESET = '\033[0m'
RED = '\033[31m'
GREEN = '\033[32m'
BLUE = '\033[34m'
MAGENTA = '\033[35m'
CYAN = '\033[36m'

def prRed(skk): 
    print("\033[91m {}\033[00m" .format(skk)) 
def prGreen(skk): 
    print("\033[92m {}\033[00m" .format(skk))   
def prMAGENTA(skk):
    print("\033[35m {}\033[00m" .format(skk)) 
def prYellow(skk):
    print("\033[93m {}\033[00m" .format(skk))   

    
# ====================================================================== #
#                         Utility / Setup
# ====================================================================== #

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================================
#                           Data & Transforms
#               Setting mean/std between CIFAR-10 & CIFAR-100
#    Normallizing inpits accelerates convergence and stabilizes training
# ============================================================================
CIFAR_MEAN = {
    "CIFAR10":  [0.4914, 0.4822, 0.4465],
    "CIFAR100": [0.5071, 0.4867, 0.4408],
}[DATASET_NAME]
CIFAR_STD = {
    "CIFAR10":  [0.2470, 0.2435, 0.2616],
    "CIFAR100": [0.2675, 0.2565, 0.2761],
}[DATASET_NAME]


# ====================================================================== #
#                    CIFAR-10 Dirichlet Partition
# ====================================================================== #

def dirichlet_partition(
    targets: np.ndarray,
    num_clients: int,
    alpha: float
):
    """
    Returns a list of index lists: indices_per_client[i] are the sample indices
    owned by client i, using a Dirichlet(alpha) distribution over clients
    for each class.
    """
    num_classes = int(targets.max()) + 1
    indices_per_client = [[] for _ in range(num_clients)]
    all_indices = np.arange(len(targets))

    for c in range(num_classes):
        class_idx = all_indices[targets == c]
        np.random.shuffle(class_idx)

        # Dirichlet over clients for this class
        proportions = np.random.dirichlet(alpha * np.ones(num_clients))
        proportions = (np.cumsum(proportions) * len(class_idx)).astype(int)
        splits = np.split(class_idx, proportions[:-1])

        for client_id, split in enumerate(splits):
            indices_per_client[client_id].extend(split.tolist())

    for cid in range(num_clients):
        np.random.shuffle(indices_per_client[cid])

    return indices_per_client


def build_dataloaders_cifar10_dirichlet(
    num_clients: int,
    alpha: float,
    batch_size: int,
    num_workers: int
):
    """
    Download CIFAR-10, create non-IID Dirichlet splits, return:
        - client_train_loaders: list[DataLoader]
        - test_loader: DataLoader
        - num_samples_per_client: list[int]
    """
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    # ============================================================================
    #                               DataLoders
    # ============================================================================
    # Download CIFAR datasets
    if DATASET_NAME == "CIFAR10":
        train_dataset = datasets.CIFAR10(root=DATA_ROOT, train=True,  download=True, transform=transform_train)
        test_dataset  = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=transform_test)
        NUM_CLASSES = 10
    else:
        train_dataset = datasets.CIFAR100(root=DATA_ROOT, train=True,  download=True, transform=transform_train)
        test_dataset  = datasets.CIFAR100(root=DATA_ROOT, train=False, download=True, transform=transform_test)
        NUM_CLASSES = 100



    targets = np.array(train_dataset.targets)
    indices_per_client = dirichlet_partition(targets, num_clients, alpha)

    client_loaders = []
    num_samples_per_client = []
    for cid in range(num_clients):
        subset = Subset(train_dataset, indices_per_client[cid])
        loader = DataLoader(
            subset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=True,
        )
        client_loaders.append(loader)
        num_samples_per_client.append(len(subset))

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    return client_loaders, test_loader, num_samples_per_client


# ====================================================================== #
#                             Model
# ====================================================================== #

def get_resnet18_cifar10():
    """
    Standard ResNet-18 for CIFAR-10 using torchvision's ImageNet variant.
    We just override the final FC layer to 10 classes.
    """
    model = models.resnet18(weights=None)
    if DATASET_NAME == "CIFAR10":
        model.fc = nn.Linear(model.fc.in_features, 10)
    else:
        model.fc = nn.Linear(model.fc.in_features, 100)
    return model


# ====================================================================== #
#                      Orthogonal Subspace Utilities
# ====================================================================== #

def init_orth_basis(model: nn.Module):
    """
    Create a map: param_name -> basis matrix (or None)

    We track only trainable parameters with dim > 1:
    (weights, BN weights, etc.; biases and running stats are ignored).
    """
    basis = {}
    for name, p in model.named_parameters():
        if p.requires_grad and p.dim() > 1:
            basis[name] = None
    return basis


def update_basis_from_delta(
    basis_map: dict,
    delta_state: dict,
    model: nn.Module,
    top_r: int,
    device: torch.device,
    eps: float = 1e-8,
):
    """
    Warm-up phase: update orth basis O_ℓ per layer using global update δ_ℓ.

    For each tracked param:
        - Orthogonalize new direction against existing basis (Gram–Schmidt)
        - Add direction up to TOP_R
    """
    with torch.no_grad():
        for name, p in model.named_parameters():
            if not (p.requires_grad and p.dim() > 1):
                continue
            if name not in delta_state:
                continue

            delta = delta_state[name].detach().to("cpu")
            v = delta.view(-1)

            if v.norm() < eps:
                continue

            B = basis_map[name]
            
            if B is None:
                v_norm = v / (v.norm() + eps)
                basis_map[name] = v_norm.view(-1, 1)  # stored on CPU
                continue

            B = B.to("cpu")

            proj_coeffs = B.t().matmul(v)   # (k)
            proj = B.matmul(proj_coeffs)    # (d)
            v_perp = v - proj


            
            if v_perp.norm() < eps:
                continue

            v_perp = v_perp / (v_perp.norm() + eps)

            
            new_basis = torch.cat([B, v_perp.view(-1, 1)], dim=1)
            if new_basis.size(1) > top_r:
                new_basis = new_basis[:, 1:]  # drop oldest

            
            q, _ = torch.linalg.qr(new_basis, mode="reduced")
            basis_map[name] = q


def project_delta_with_basis(
    delta_state: dict,
    basis_map: dict,
    model: nn.Module,
    device: torch.device,
    eps: float = 1e-8,
):
    """
    For each tracked param, project δ_ℓ onto orthogonal complement of O_ℓ:

        δ_ℓ^⊥ = (I - O_ℓ O_ℓ^T) δ_ℓ

    Returns a new state dict of projected deltas.
    """
    projected = {}
    with torch.no_grad():
        # Clones - Keep original device for non-FOT tensors
        for key, tensor in delta_state.items():
            projected[key] = tensor.clone()

        for name, p in model.named_parameters():
            if not (p.requires_grad and p.dim() > 1):
                continue
            if name not in delta_state:
                continue

                
            B = basis_map.get(name, None)
            if B is None:
                continue

            # delta is whatever device it's currently on (either GPU or CPU)
            delta = delta_state[name]
            org_device = delta.device


            # Move to CPU for projection
            v = delta.detach().to("cpu").view(-1)
            B_cpu = B.to("cpu")

            if v.norm() < eps:
                continue

            coeffs = B_cpu.t().matmul(v)        # (k)
            proj = B_cpu.matmul(coeffs)         # (d)
            v_perp = v - proj                   # orthogonal component

            v_perp = v_perp.view_as(delta.cpu())

            # Move back to original device to stay with the rest of state
            projected[name] = v_perp.to(org_device)

    return projected


# ====================================================================== #
#                       Local Client Training
# ====================================================================== #

def get_cosine_lr(base_lr: float, cur_round: int, total_rounds: int):
    if SCHEDULER.lower() != "cosine":
        return base_lr
    if total_rounds <= 0:
        return base_lr
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * cur_round / total_rounds))


def make_criterion():
    if LABEL_SMOOTH > 0.0:
        return nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    return nn.CrossEntropyLoss()


def local_train(
    model: nn.Module,
    train_loader: DataLoader,
    device: torch.device,
    lr: float,
    local_epochs: int,
    momentum=MOMENTUM, 
    weight_decay=WEIGHT_DECAY,
    client_id=None
):
    """
    One client's local SGD. Returns the updated model (in-place) and
    the number of training samples.
    """
    model.to(device)
    model.train()

    criterion = make_criterion().to(device)
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    total_loss, total_cnt, correct = 0.0, 0, 0

    num_samples = 0
    for ep in range(LOCAL_EPOCHS):
        # per-local-epoch accumulators
        ep_loss, ep_cnt, ep_correct = 0.0, 0, 0
        
        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            num_samples += x.size(0)

            bs = y.size(0)
            total_loss += loss.item() * bs
            total_cnt  += bs
            correct    += (logits.argmax(1) == y).sum().item()

            ep_loss    += loss.item() * bs
            ep_cnt     += bs
            ep_correct += (logits.argmax(1) == y).sum().item()

        acc_avg_train  = 100.0 * ep_correct / max(1, ep_cnt)
        loss_avg_train = ep_loss / max(1, ep_cnt)
        
        
        prRed(" Client{} Train => Local Epoch: {} \tAcc: {:.3f} \tLoss: {:.4f}".format(
            (client_id if client_id is not None else "?"), ep, acc_avg_train, loss_avg_train
        ))

        
    return model, num_samples


# ====================================================================== #
#                        FedAvg Aggregation
# ====================================================================== #

def fedavg_aggregate(
    global_state: dict,
    local_states: list,
    sample_counts: list,
    device: torch.device,
):
    """
    Standard FedAvg: weighted average of local models by number of samples.

    We only average *floating-point* tensors (parameters/buffers).
    Non-floating tensors (like num_batches_tracked) are copied from the
    global_state and their delta is set to zero.
    """
    assert len(local_states) == len(sample_counts)
    total_samples = float(sum(sample_counts))

    new_state = {}

    # Initialize new_state: floats -> zeros, others -> copy global
    for key, g_tensor in global_state.items():
        if g_tensor.dtype.is_floating_point:
            new_state[key] = torch.zeros_like(g_tensor, device=device)
        else:
            new_state[key] = g_tensor.to(device)

    # Weighted aggregation for floats only
    for state, n_i in zip(local_states, sample_counts):
        w = n_i / total_samples
        for key, local_tensor in state.items():
            g_tensor = global_state[key]
            if not g_tensor.dtype.is_floating_point:
                continue
            new_state[key] += local_tensor.to(device) * w

    # Compute delta only for floats; others set to zero
    delta_state = {}
    for key, g_tensor in global_state.items():
        if g_tensor.dtype.is_floating_point:
            delta_state[key] = new_state[key] - g_tensor.to(device)
        else:
            delta_state[key] = torch.zeros_like(g_tensor, device=device)

    return new_state, delta_state


# ====================================================================== #
#                           Evaluation
# ====================================================================== #

def evaluate(model: nn.Module, test_loader: DataLoader, device: torch.device):
    model.to(device)
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0.0
    criterion = nn.CrossEntropyLoss().to(device)

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss_sum += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += x.size(0)

    acc = correct / total * 100.0
    loss_mean = loss_sum / total
    
    return acc, loss_mean


def evaluate_train_global(model: nn.Module, client_loaders, device: torch.device):
    """
    Evaluate global model on union of all client training data.
    Returns (acc_percent, mean_loss).
    """
    model.to(device)
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0.0
    criterion = nn.CrossEntropyLoss().to(device)

    with torch.no_grad():
        for loader in client_loaders:
            for x, y in loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                logits = model(x)
                loss = criterion(logits, y)
                loss_sum += loss.item() * x.size(0)

                preds = logits.argmax(dim=1)
                correct += (preds == y).sum().item()
                total += x.size(0)

    acc = correct / total * 100.0
    mean_loss = loss_sum / total
    return acc, mean_loss


def make_fgt_windows(test_loader, num_windows: int):
    """
    Split the test dataset into num_windows contiguous windows and return
    a list of DataLoaders (one per window).
    """
    test_dataset = test_loader.dataset
    N = len(test_dataset)
    indices = np.arange(N)
    windows = np.array_split(indices, num_windows)

    window_loaders = []
    for w_idx in windows:
        subset = Subset(test_dataset, w_idx.tolist())
        loader = DataLoader(
            subset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True,
        )
        window_loaders.append(loader)
    return window_loaders


def eval_loss_on_window(model: nn.Module, loader: DataLoader, device: torch.device):
    """
    Mean CE loss on a single window (DataLoader).
    """
    model.to(device)
    model.eval()
    ce = nn.CrossEntropyLoss(reduction="sum").to(device)
    total = 0
    loss_sum = 0.0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            loss = ce(logits, y)
            loss_sum += loss.item()
            total += x.size(0)

    if total == 0:
        return 0.0
    return loss_sum / total


# ====================================================================== #
#                               Main
# ====================================================================== #
set_seed(SEED)
device = get_device()
print(f"Using device: {device}")

print("Preparing CIFAR-10 loaders (Dirichlet non-IID, alpha = {:.3f})...".format(ALPHA))
client_loaders, test_loader, num_samples_per_client = build_dataloaders_cifar10_dirichlet(
    NUM_CLIENTS, ALPHA, BATCH_SIZE, NUM_WORKERS
)
print("Client sample counts:", num_samples_per_client)

# Create forgetting windows over the test set
fgt_window_loaders = make_fgt_windows(test_loader, NUM_FGT_WINDOWS)
# Historical best loss per window (initialized to +inf)
best_loss_per_window = [float("inf")] * NUM_FGT_WINDOWS

print("Building ResNet-18 model...")
global_model = get_resnet18_cifar10().to(device)
global_state = deepcopy(global_model.state_dict())

# Orthogonal basis map
orth_basis = init_orth_basis(global_model)

# ---------------------- History logs ---------------------- #
acc_train_hist = []
acc_test_hist = []
round_time_hist = []
mean_fgt_hist = []      # window-based forgetting per round

num_clients_per_round = max(1, int(FRAC * NUM_CLIENTS))
print(f"Clients per round: {num_clients_per_round} / {NUM_CLIENTS}")
print(f"EPOCHS (global rounds): {EPOCHS}, LOCAL_EPOCHS: {LOCAL_EPOCHS}")
print(f"ORTH = {ORTH}, TOP_R = {TOP_R}, WARMUP_ROUNDS = {WARMUP_ROUNDS}")
print(f"NUM_FGT_WINDOWS = {NUM_FGT_WINDOWS}")
print("-" * 80)

for rnd in range(EPOCHS):
    round_start = time.time()

    if cosine_schedule:
        lr = get_cosine_lr(LR_BASE, rnd, EPOCHS - 1)
    else:
        lr = LR_BASE
    
    print(f"\n=== Round {rnd+1}/{EPOCHS} | LR = {lr:.4f} ===")
    m = max(1, int(np.ceil(FRAC * NUM_CLIENTS)))
    selected = np.random.choice(NUM_CLIENTS, m, replace=False)

    # # Sample clients (FRAC=1.0 → all, but kept general)
    # all_client_ids = list(range(NUM_CLIENTS))
    # random.shuffle(all_client_ids)
    # selected = all_client_ids[:num_clients_per_round]
    # print(f"Selected clients: {selected}")

    local_states = []
    sample_counts = []
    

    # Local training
    for cid in selected:
        prGreen(f" Client {cid}")
        client_model = deepcopy(global_model)
        client_loader = client_loaders[cid]
        client_model, n_i = local_train(
            client_model,
            client_loader,
            device,
            lr,
            LOCAL_EPOCHS,
            MOMENTUM,
            WEIGHT_DECAY,
            int(cid)
        )
        local_states.append(deepcopy(client_model.state_dict()))
        sample_counts.append(n_i)


    # FedAvg aggregation
    new_state, delta_state = fedavg_aggregate(
        global_state,
        local_states,
        sample_counts,
        device,
    )

    # FOT-style orthogonal processing of the global update
    if ORTH:
        if rnd < WARMUP_ROUNDS:
            print("Warm-up phase: updating orth basis (no projection yet).")
            update_basis_from_delta(
                orth_basis,
                delta_state,
                global_model,
                top_r=TOP_R,
                device=device,
            )
            effective_delta = delta_state
        else:
            print("FOT phase: projecting global update onto orthogonal complement.")
            projected_delta = project_delta_with_basis(
                delta_state,
                orth_basis,
                global_model,
                device=device,
            )
            effective_delta = projected_delta
            for key in new_state.keys():
                new_state[key] = global_state[key].to(device) + effective_delta[key]
    else:
        effective_delta = delta_state
    

    # Update global model
    global_state = {k: v.clone().detach() for k, v in new_state.items()}
    global_model.load_state_dict(global_state)

    # ----------------- Evaluate on train & test ----------------- #
    train_acc, train_loss = evaluate_train_global(global_model, client_loaders, device)
    acc_train_hist.append(train_acc)

    test_acc, test_loss = evaluate(global_model, test_loader, device)
    acc_test_hist.append(test_acc)

    round_time = time.time() - round_start
    round_time_hist.append(round_time)

    
    # ----------------- Window-based forgetting (signed) ----------------- #
    # Implements the definition from your slide:
    #   FgT_t^(τ) = L_now - best_loss_per_window[τ]
    #   negative  => improvement
    #   positive  => forgetting
    fgt_list = []

    for w_idx, loader_w in enumerate(fgt_window_loaders):
        L_cur = eval_loss_on_window(global_model, loader_w, device)
        if not math.isfinite(L_cur):
            continue

        # First time we see this window: initialize baseline, no forgetting yet
        if best_loss_per_window[w_idx] == float("inf"):
            best_loss_per_window[w_idx] = L_cur
            fgt = 0.0
        else:
            # FgT_t^(τ) = L_now - best_loss_per_window[τ]
            fgt = L_cur - best_loss_per_window[w_idx]

            # Update historical best if we just improved
            if L_cur < best_loss_per_window[w_idx]:
                best_loss_per_window[w_idx] = L_cur

        fgt_list.append(fgt)

    # Per-round mean forgetting over all windows
    mean_fgt = float(np.mean(fgt_list)) if len(fgt_list) > 0 else 0.0
    mean_fgt_hist.append(mean_fgt)


    print(
        f"Round {rnd+1:02d}: "
        f"Train Acc = {train_acc:.2f}% | "
        f"Test Acc = {test_acc:.2f}% | "
        f"Test Loss = {test_loss:.4f} | "
        f"Mean FgT = {mean_fgt:.4f} | "
        f"Time = {round_time:.2f}s"
    )

print("\nTraining finished.")
final_acc, final_loss = evaluate(global_model, test_loader, device)
print(f"Final Test Accuracy: {final_acc:.2f}% | Loss: {final_loss:.4f}")

# =================================================================
#                              Save logs
# =================================================================
n = min(
    len(acc_train_hist),
    len(acc_test_hist),
    len(round_time_hist),
    len(mean_fgt_hist),
)
round_idx = list(range(1, n + 1))

df = pd.DataFrame({
    "round":          round_idx,
    "acc_train":      acc_train_hist[:n],
    "acc_test":       acc_test_hist[:n],
    "round_time_sec": round_time_hist[:n],
    "mean_FgT_loss":  mean_fgt_hist[:n],
    "projection":     [ORTH] * n,
    "alpha":          [ALPHA] * n,
    "warmup_rounds":  [WARMUP_ROUNDS] * n,
    "rebuild_every":  [REBUILD_EVERY] * n,
    "top_r":          [TOP_R] * n,
})

csv_name = f"{PROGRAM_NAME}.csv"
df.to_csv(csv_name, index=False)
print("Saved:", csv_name)



Using constant LR = 0.01 (cosine_schedule=False)
 	Data:		CIFAR10
	Cut:		Original
	Global Rounds:	100
	Clients:	10
	Local Epochs:	5
	α:		0.5
	Warm-up:	5
	Rebuild:	N/A
Using device: cuda
Preparing CIFAR-10 loaders (Dirichlet non-IID, alpha = 0.500)...
Client sample counts: [7712, 3982, 3834, 3985, 7708, 4771, 3729, 3399, 4634, 6246]
Building ResNet-18 model...
Clients per round: 10 / 10
EPOCHS (global rounds): 100, LOCAL_EPOCHS: 5
ORTH = True, TOP_R = 8, WARMUP_ROUNDS = 5
NUM_FGT_WINDOWS = 5
--------------------------------------------------------------------------------

=== Round 1/100 | LR = 0.0100 ===
  Client 5
  Client5 Train => Local Epoch: 0 	Acc: 35.339 	Loss: 1.7729
  Client5 Train => Local Epoch: 1 	Acc: 46.783 	Loss: 1.4434
  Client5 Train => Local Epoch: 2 	Acc: 53.217 	Loss: 1.2649
  Client5 Train => Local Epoch: 3 	Acc: 55.460 	Loss: 1.1858
  Client5 Train => Local Epoch: 4 	Acc: 61.035 	Loss: 1.0805
  Client 1
  Client1 Train => Local Epoch: 0 	Acc: 32.521 	Loss: 1.9832
